In [1]:
import pandas as pd
from load import load_dataset

dataset = load_dataset("futbol_uruguayo.csv")

print(dataset.head())


TypeError: load_attributes.<locals>.get_record() missing 1 required positional argument: 'team'

In [ ]:
#separamos cronologicamente el conjunto de entrenamiento y el de evaluacion
#la evaluacion se mantiene separada hasta haber elegido los hiperparametros

train = dataset[
    dataset["date"] < pd.Timestamp("2024-01-01")
].copy()

test = dataset[
    (dataset["date"] >= pd.Timestamp("2024-01-01"))
    & (dataset["date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Cantidad de partidos de entrenamiento:", len(train))
print("Cantidad de partidos de evaluacion:", len(test))

In [ ]:
import sys
import os

from pipeline import create_model_pipeline, pipeline_input_attributes


# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#hacemos el arbol de decision usando solamente el conjunto de entrenamiento
from naiveBayes.bayes import M_Estimator as BayesClassifier

bayes= BayesClassifier(m=1.0)

model = create_model_pipeline(bayes)

In [ ]:
X_train = train[pipeline_input_attributes].copy()
y_train = train["result"].copy()

X_test = test[pipeline_input_attributes].copy()
y_test = test["result"].copy()

print("Filas de entrenamiento:", len(X_train))
print("Filas reservadas para evaluacion final:", len(X_test))

In [ ]:
import numpy as np

#cada temporada se valida usando solamente las temporadas anteriores
validation_years = [2020, 2021, 2022, 2023]
train_years = train["date"].dt.year.to_numpy()
temporal_splits = []

for validation_year in validation_years:
    fit_indices = np.flatnonzero(
        train_years < validation_year
    )
    validation_indices = np.flatnonzero(
        train_years == validation_year
    )

    temporal_splits.append((
        fit_indices,
        validation_indices
    ))

    print(
        f"Validacion {validation_year}:",
        f"entrenamiento={len(fit_indices)},",
        f"validacion={len(validation_indices)}"
    )

In [ ]:
from sklearn.model_selection import GridSearchCV

#en esta primera busqueda variamos solamente el margen de record
param_grid = {
    #diferencia de tasa de victorias
    "preprocessing__differences__discretizer__record_margin": [
        0.00,
        0.025,
        0.05,
    ],

    #diferencia de forma reciente
    "preprocessing__differences__discretizer__last_matches_margin": [
        0.00,
        0.07,
        0.14,
    ],

    #diferencia de goles general
    "preprocessing__differences__discretizer__goal_difference_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #diferencia de goles convertidos
    "preprocessing__differences__discretizer__attack_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #diferencia de goles recibidos
    "preprocessing__differences__discretizer__defense_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #valor de m para el m-estimador
    "model__m": [
        0.00,
        0.10,
        0.50,
        1.00,
        2.00,
        5.00,
        10.00,
        20.00,
    ],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=temporal_splits,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparametros:")
print(grid_search.best_params_)
print(f"Mejor accuracy temporal: {grid_search.best_score_:.3%}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)

#buscamos automaticamente todas las columnas de hiperparametros
parameter_columns = [
    column
    for column in results.columns
    if column.startswith("param_")
]

score_columns = [
    "split0_test_score",
    "split1_test_score",
    "split2_test_score",
    "split3_test_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]

results = results[
    parameter_columns + score_columns
].sort_values("rank_test_score")

results = results.rename(columns={
    "split0_test_score": "accuracy_2020",
    "split1_test_score": "accuracy_2021",
    "split2_test_score": "accuracy_2022",
    "split3_test_score": "accuracy_2023",
})

results.head(10)